# Multi-Observation Grism Fitting Demo: New Code Structure

This notebook mirrors `multi_obs_demo.ipynb` but explains the restructured architecture.
Read `simple_fit_demo_newstructure.ipynb` first for an overview of the three-group parameter
structure and the new `FitConfiguration` API.

## Multi-observation specifics

In the multi-obs fit, **shared parameters** (morphology + geometry + rotation curve shape)
are sampled once across all observations. **Per-observation parameters** (`v0`, `amplitude`)
are sampled independently for each observation.

```
Shared (sampled once):
  morph:   amplitude*, r_eff, n, PA_morph, xc_morph, yc_morph
  geom:    PA, i, sigma0, x0_vel*, y0_vel*
  rot:     Va, r_t  (or log_M_star, log_M_halo, c_halo for mass models)

Per-observation (sampled independently):
  v0_obs1, v0_obs2, ...           (systemic velocity per grism)
  amplitude_obs1, amplitude_obs2, ...  (flux normalisation per grism)
```

`*` amplitude and x0_vel/y0_vel are actually shared in the multi-obs model;
only `v0` and `amplitude` are per-obs.

In [ ]:
from geko.fitting import run_geko_fit_multi, run_geko_fit
from geko.config import FitConfiguration, MCMCSettings

import jax
import numpyro
jax.config.update('jax_enable_x64', True)
numpyro.set_host_device_count(2)

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Step 1: Data parameters

The `field` parameter auto-selects the grism file naming convention, PSF, and default
rotation angle for the supported JWST grism surveys:

| `field` | Programme | Auto-set PSF | Auto-set `theta_rot` |
|---|---|---|---|
| `'GOODS-S-FRESCO'` | FRESCO (PID 1895), GOODS-S | `mpsf_jw018950.gs.f444w.fits` | 0° |
| `'GOODS-N'` | FRESCO (PID 1895), GOODS-N | `mpsf_jw018950.gn.f444w.fits` | 230.5° |
| `'GOODS-N-CONGRESS'` | CONGRESS (PID 3577), GOODS-N | `mpsf_jw035770.f356w.fits` | 228.2° |
| `'manual'` | Any | `manual_psf_name` | `manual_theta_rot` per observation |

In multi-observation mode the single `field` value drives preprocessing (PSF, file naming)
for all observations. Each observation's own `theta_rot` is then specified individually
in `observations_config` — see Step 2.

In [ ]:
geko_path = '/Users/lola/geko/'   # change to your path

source_id            = 191250
field                = 'manual'
output_name          = 'my_galaxy'
master_catalog       = geko_path + 'demo/simple_fit_demo_files/catalogs/my_galaxies_cat'
emission_line        = 'H_alpha'   # emission line name — used to look up the line in the master catalog
parametric           = True
save_runs_path       = geko_path + 'demo/simple_fit_demo_files/'

manual_psf_name      = 'webbPSF_F444W.fits'
manual_pysersic_file = 'summary_191250_image_F150W_svi.cat'
grism_file           = 'spec_2d_FRESCO_F444W_ID191250_comb.fits'

grism_filter         = 'F444W'
delta_wave_cutoff    = 0.02
factor               = 1     # set low for demo speed (use 5 in real runs)
wave_factor          = 1     # set low for demo speed (use 9 in real runs)

num_chains  = 1
num_warmup  = 50
num_samples = 50

## Step 2: Define observations

Each observation is a dict with four keys:

| Key | Type | Description |
|---|---|---|
| `grism_file` | str | FITS filename, looked up as `<save_runs_path>/<output>/<filename>` |
| `theta_rot` | float | Rotation angle in **degrees** between this grism observation and the reference imaging frame |
| `dispersion` | str | `'R'` — wavelength dispersed along rows; `'C'` — along columns |
| `name` | str | Label used in output filenames and summary plot rows |

### `theta_rot`: aligning grism and imaging frames

`theta_rot` is the rotation angle (degrees) between a grism observation's dispersion axis
and the NIRCam imaging frame used for morphology.

geko models the galaxy from a direct image (via PySersic), then projects that morphology
into each grism frame to generate a predicted 2D spectrum. When a grism and the imaging
were observed at different telescope position angles, the morphological PA must be rotated
before projection. `theta_rot` encodes that relative orientation for each grism.

**In single-observation fitting** there is one `theta_rot` for the whole run.

**In multi-observation fitting** each observation has its own `theta_rot`, and this is
the key that allows geko to fit grism spectra taken at different observing position
angles simultaneously. For example, if two visits were separated by a 90° roll:

```
obs1: theta_rot =  0°   → morphological PA in grism 1 frame = PA_morph - 0°
obs2: theta_rot = 90°   → morphological PA in grism 2 frame = PA_morph - 90°
```

The galaxy parameters (PA, inclination, rotation curve) are shared; `theta_rot` is a
**fixed per-observation input**, not a fitted parameter.

For `field='manual'`, compute it from your FITS headers:
```
theta_rot = PA_V3_grism - PA_V3_imaging   (mod 360°)
```
where `PA_V3` (or `ROLL_REF`) is in the primary FITS header of each observation.
For the predefined fields this angle is hardcoded from the known survey orientations
relative to JADES NIRCam imaging.

This demo uses the same file twice at `theta_rot = 0°`. In a real run use observations
at distinct position angles with their measured `theta_rot` values.

In [ ]:
observations_config = [
    {
        'grism_file': grism_file,
        'theta_rot':  0.0,    # rotation angle in degrees
        'dispersion': 'R',    # R (row) or C (column)
        'name':       'obs1'
    },
    {
        'grism_file': grism_file,   # same file for demo
        'theta_rot':  0.0,
        'dispersion': 'R',
        'name':       'obs2'
    },
]

print(f'{len(observations_config)} observations configured')
for obs in observations_config:
    print(f"  {obs['name']}: {obs['grism_file']}  θ={obs['theta_rot']}°  {obs['dispersion']}")

## Step 3: Build the configuration

The config API is identical to the single-obs case. The same override dicts apply;
`run_geko_fit_multi` uses them the same way as `run_geko_fit`.

In [ ]:
fit_config = FitConfiguration(
    mcmc=MCMCSettings(
        num_chains=num_chains,
        num_warmup=num_warmup,
        num_samples=num_samples,
    ),
    # Morphology priors: param names are amplitude, r_eff, n, PA_morph, xc_morph, yc_morph
    # PySersic values are loaded first; anything here overrides them.
    morph_prior_overrides={
        'r_eff_max': 12.0,
    },
    # Geometry / shared kinematics: PA, i, sigma0, x0_vel, y0_vel, v0
    geom_prior_overrides={
        'i_min':      20.0,
        'i_max':      80.0,
        'sigma0_min':  0.0,
        'sigma0_max': 200.0,
    },
    # Rotation curve priors: Va, r_t (Arctan)
    rot_prior_overrides={
        'Va_min': -1000.0,
        'Va_max':  1000.0,
    },
)
fit_config.print_summary()

## Step 4: Run multi-observation fitting

The call is unchanged from the old demo. Internally `run_geko_fit_multi` now:
1. Builds the morphology model (`SersicMorphology`) and rotation model (`CompositeRotationCurve([ArctanComponent()])`)
2. Loads PySersic priors, then applies config overrides
3. Runs MCMC with shared parameters and per-obs `v0` / `amplitude`
4. Post-processes: computes v_re, v/σ, writes results table, generates summary plot

In [ ]:
inf_data, results = run_geko_fit_multi(
    observations_config=observations_config,
    output=output_name,
    master_cat=master_catalog,
    line=emission_line,
    parametric=parametric,
    save_runs_path=save_runs_path,
    num_chains=num_chains,
    num_warmup=num_warmup,
    num_samples=num_samples,
    source_id=source_id,
    field=field,
    grism_filter=grism_filter,
    delta_wave_cutoff=delta_wave_cutoff,
    factor=factor,
    wave_factor=wave_factor,
    config=fit_config,
    manual_psf_name=manual_psf_name,
    manual_pysersic_file=manual_pysersic_file,
)

print('\nFitting complete!')
print(f'Results available for: {list(results.keys())}')

## Step 5: Inspect posterior results

Results are stored in structured dicts — no more scattered named attributes.

In [ ]:
import os
import arviz as az
from astropy.table import Table

output_dir = os.path.join(save_runs_path, output_name)
results_file = os.path.join(output_dir, f'{source_id}_results_multi')
if os.path.exists(results_file):
    fit_results = Table.read(results_file, format='ascii')
    print(fit_results)
else:
    print(f'Results file not found: {results_file}')

In [ ]:
# Shared rotation curve parameters — model-agnostic dict
# (works for Arctan now; will work for Sersic+NFW without any code change)
print('Rotation curve posterior medians:')
# These live in inf_data.posterior — look for any non-'unscaled_' parameter
shared_params = [v for v in inf_data.posterior.data_vars if not v.startswith('unscaled_')]
for p in sorted(shared_params):
    med = float(inf_data.posterior[p].median())
    print(f'  {p:20s}: {med:.3f}')

In [ ]:
# Per-observation parameters (v0 and amplitude sampled independently per obs)
print('Per-observation parameters:')
for obs in observations_config:
    name = obs['name']
    for key in [f'v0_{name}', f'amplitude_{name}']:
        if key in inf_data.posterior:
            med = float(inf_data.posterior[key].median())
            print(f'  {key:30s}: {med:.4f}')

## Step 6: Output files

| File | Contents |
|---|---|
| `{source_id}_output_multi` | Full MCMC posterior (arviz InferenceData, NetCDF) |
| `{source_id}_results_multi` | Median ± 1σ + derived quantities (v_re, v/σ) |
| `{source_id}_summary_multi.png` | N-row summary plot (one row per observation + intrinsic fields row) |
| `{source_id}_summary_corner_multi.png` | Full corner plot for shared parameters |

In [ ]:
from IPython.display import Image, display

summary_plot = os.path.join(output_dir, f'{source_id}_summary_multi.png')
if os.path.exists(summary_plot):
    display(Image(filename=summary_plot, width=900))
else:
    print(f'Plot not found: {summary_plot}')
    print('Available files in output dir:')
    print([f for f in os.listdir(output_dir) if f.endswith('.png')])

In [ ]:
corner_plot = os.path.join(output_dir, f'{source_id}_summary_corner_multi.png')
if os.path.exists(corner_plot):
    display(Image(filename=corner_plot, width=800))

## Step 7: MCMC diagnostics

In [ ]:
import matplotlib.pyplot as plt

az.plot_trace(inf_data, var_names=['Va', 'sigma0', 'PA', 'i'])
plt.tight_layout()
plt.show()

## Switching to a composite mass-based rotation model

Once `SersicComponent` and `NFWComponent` are implemented (Step 8 of the refactor plan),
switching is a one-line config change — no other code needs to change:

```python
fit_config = FitConfiguration(
    rotation_components=['Sersic', 'NFW'],
    rot_prior_overrides={
        'log_M_star_min': 9.0,
        'log_M_star_max': 12.0,
        'log_M_halo_min': 10.0,
        'log_M_halo_max': 14.0,
        'c_halo_min': 1.0,
        'c_halo_max': 30.0,
    },
    mcmc=MCMCSettings(...),
)
```

The total rotation velocity will be `v(r) = sqrt(v²_Sersic(r) + v²_NFW(r))` automatically.
The results table and corner plot will include `log_M_star_50`, `log_M_halo_50`, `c_halo_50`
without any changes to postprocessing or plotting code.

## Summary of what changed

| Old | New |
|---|---|
| `MorphologyPriors(PA_mean=..., inc_mean=...)` | `geom_prior_overrides={'PA_mu': ..., 'i_mu': ...}` |
| `KinematicPriors(Va_max=...)` | `rot_prior_overrides={'Va_max': ...}` |
| `FitConfiguration(morphology=MorphologyPriors(...), kinematics=KinematicPriors(...))` | `FitConfiguration(morph_prior_overrides={}, geom_prior_overrides={}, rot_prior_overrides={})` |
| Hardcoded arctan velocity call in inference | `galaxy_model.velocity_field(X, Y, PA, i, all_params)` — any rotation model |
| `Va_mean`, `r_t_mean` attrs | `kin_model.rot_means['Va']`, `kin_model.rot_means['r_t']` |
| `r_eff_mean`, `n_mean` etc. | `kin_model.morph_means['r_eff']` etc. |